# Лабораторна 6. Власна RAG-система над документацією FastAPI

Цей ноутбук — виконання пункту 1 «Власна RAG-система»: корпус документів, бенчмарк, індексація, оцінювання ретриву, генерація з цитуванням, аналіз відмов, демонстрація і звіт. Розділи 1.1–1.8 відповідають розділам методички, звіт — окремою секцією в кінці.

## 1.1. Корпус

**Корпус** — офіційна англомовна документація Python-фреймворку **FastAPI** (розділи Tutorial, Advanced User Guide, Deployment, How-To та кілька оглядових сторінок: async, python-types, features, fastapi-cli, environment-variables, virtual-environments, alternatives, benchmarks, history-design-future).

- **Джерело:** репозиторій https://github.com/fastapi/fastapi, каталог docs/en/docs. Версію зафіксовано за хешем коміту, тому корпус і бенчмарк відтворювані.
- **Ліцензія:** MIT (Copyright © 2018 Sebastián Ramírez), тож використання й цитування дозволені. У репозиторій лабораторної корпус не додається: він завантажується в коді (git sparse checkout за хешем коміту).
- **Формат:** Markdown (.md). Більшість прикладів коду підключається в .md-файлах директивами на файли з docs_src, тому ми підставляємо вміст прикладів у текст сторінки. Службову розмітку (admonitions, якорі заголовків, картинки, HTML) прибираємо.
- **Кількість документів і обсяг** — див. вивід клітинки нижче.

**Хто й навіщо ставитиме питання.** Розробник, який вчить FastAPI або пише на ньому застосунок (у моєму випадку це дипломний проєкт). Питання виду «який параметр задає X», «яка команда запускає Y», «чим A відрізняється від B». Відповідь потрібна швидко, з посиланням на сторінку документації.

**Чому саме RAG, а не промпт чи донавчання:**
- *Промпт:* вся документація — це близько 250 тисяч токенів. Вона не поміщається в контекст невеликої моделі, а в великий контекст її кожного разу довелося б передавати повністю (дорого й повільно), і якість відповідей у наддовгому контексті падає.
- *Донавчання:* документація змінюється з кожним релізом (наприклад, у ній описано змінну FASTAPI_ENV, а рекомендована бібліотека для хешування паролів змінилася з passlib на pwdlib). Донавчання довелося б повторювати, воно не дає посилань на джерело і не гарантує відсутності галюцинацій.
- *RAG:* індекс оновлюється за хвилини, відповідь спирається на конкретні фрагменти, можна показати джерело і відмовитися, коли відповіді в документації немає.

**Зв'язок із реальними задачами.** Питання бенчмарку взято з типових задач мого дипломного проєкту на FastAPI: запуск і налаштування застосунку, валідація параметрів, залежності, безпека, фонові задачі, розгортання в контейнері. Це питання, які я справді шукаю в документації під час роботи.

In [ ]:
!pip install -q "sentence-transformers>=5" langchain-text-splitters faiss-cpu rank_bm25 gradio

In [ ]:
import re
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Пристрій:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(увімкніть GPU: Runtime → Change runtime type → T4 GPU)")

Пристрій: cuda Tesla T4


In [ ]:
# Завантажуємо лише потрібні каталоги репозиторію FastAPI на зафіксованому коміті (sparse checkout)
COMMIT = "05e6df7e906c3decba377c37548019973f40bf15"
script = f'''
set -e
rm -rf fastapi_src && mkdir fastapi_src && cd fastapi_src
git init -q
git remote add origin https://github.com/fastapi/fastapi.git
git sparse-checkout set docs/en/docs docs_src
git fetch -q --depth 1 --filter=blob:none origin {COMMIT}
git checkout -q FETCH_HEAD
git rev-parse HEAD
'''
result = subprocess.run(["bash", "-c", script], capture_output=True, text=True)
print(result.stdout, result.stderr[-500:])

05e6df7e906c3decba377c37548019973f40bf15
 


In [ ]:
DOCS_DIR = Path("fastapi_src/docs/en/docs")
SRC_DIR = Path("fastapi_src/docs_src")
SECTIONS = ["tutorial", "advanced", "deployment", "how-to"]
TOP_LEVEL = ["features", "python-types", "async", "fastapi-cli", "environment-variables",
             "virtual-environments", "history-design-future", "alternatives", "benchmarks"]


def list_files():
    files = [DOCS_DIR / f"{name}.md" for name in TOP_LEVEL]
    for section in SECTIONS:
        files += sorted((DOCS_DIR / section).rglob("*.md"))
    return [f for f in files if f.exists()]


def inline_code(match):
    # директива {* ../../docs_src/.../file.py hl[..] *} -> вміст файлу з прикладом
    path = SRC_DIR / match.group(1).split("docs_src/", 1)[-1]
    if not path.exists():
        return ""
    return "```python\n" + path.read_text(encoding="utf-8").strip() + "\n```"


def admonition(m):
    title = (m.group(2) or "").strip()
    return f"{m.group(1).capitalize()}{': ' + title if title else ''}:"


def clean_markdown(text):
    text = re.sub(r"\{\*\s*(\S+?)(?:\s[^\n]*?)?\s*\*\}", inline_code, text)
    text = re.sub(r"\{!>?\s*(\S+?)\s*!\}", inline_code, text)
    text = re.sub(r"\s*\{ #[\w\-.]+ \}", "", text)                      # якоря заголовків
    text = re.sub(r"^/{3,4}\s*(\w+)\s*(?:\|\s*(.*))?$", admonition, text, flags=re.M)   # /// tip | Title
    text = re.sub(r"^/{3,4}\s*$", "", text, flags=re.M)                   # закриття блоків
    text = re.sub(r"<dfn title=([\"']).*?\1>(.*?)</dfn>", r"\2", text, flags=re.S)
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", text)                      # картинки
    text = re.sub(r"<img[^>]*>", "", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)                  # посилання -> текст
    text = re.sub(r"</?(?:div|span|a|details|summary|small|sup|sub|br|p|strong|em|font|b|i|u|abbr|kbd|center|blockquote)(?:\s[^>]*)?>", "", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def page_url(rel):
    parts = list(rel.parts)
    if parts[-1] == "index":
        parts = parts[:-1]
    return "https://fastapi.tiangolo.com/" + "/".join(parts) + ("/" if parts else "")


def load_documents():
    docs = []
    for f in list_files():
        rel = f.relative_to(DOCS_DIR).with_suffix("")
        raw = re.sub(r"\A---.*?---\s*", "", f.read_text(encoding="utf-8"), flags=re.S)    # front matter
        text = clean_markdown(raw)
        m = re.search(r"^#\s+(.+)$", text, flags=re.M)
        docs.append({"doc_id": rel.as_posix(), "title": m.group(1).strip() if m else rel.name,
                     "text": text, "url": page_url(rel)})
    return docs


docs = load_documents()
total_chars = sum(len(d["text"]) for d in docs)
print(f"Документів: {len(docs)}; загальний обсяг: {total_chars:,} символів (≈ {total_chars // 4:,} токенів); середня довжина документа: {total_chars // len(docs):,} символів")
print(pd.Series([d["doc_id"].split("/")[0] if "/" in d["doc_id"] else "огляд" for d in docs]).value_counts().to_string())
print("Найдовший:", max(docs, key=lambda d: len(d["text"]))["doc_id"], "| Приклад початку:", docs[10]["doc_id"], "->", repr(docs[10]["text"][:120]))

Документів: 117; загальний обсяг: 975,996 символів (≈ 243,999 токенів); середня довжина документа: 8,341 символів
tutorial      51
advanced      36
how-to        12
огляд          9
deployment     9
Найдовший: advanced/security/oauth2-scopes | Приклад початку: tutorial/bigger-applications -> "# Bigger Applications - Multiple Files\n\nIf you are building an application or a web API, it's rarely the case that you c"


## 1.2. Бенчмарк

Бенчмарк із **28 питань** складено до налаштування системи, за самим текстом документації. Для кожного питання зафіксовано: правильні сторінки (doc_id), **регулярний вираз-доказ**, що має зустрічатися в правильному фрагменті, й еталонну відповідь.

- 24 питання мають відповідь у корпусі: пошук параметра чи команди («Яким параметром Query обмежують довжину?»), значення за замовчуванням (allow_methods), порівняння (fastapi run проти fastapi dev), пояснення («чому UploadFile зручніше за bytes»), питання з пасткою (бібліотека для хешування паролів: у тексті згадано і нову pwdlib, і старішу passlib).
- 4 питання **без відповіді в документах** (rate limiting, метрики Prometheus, MongoDB/Beanie, ціна FastAPI Cloud). Правильна поведінка системи — відмовитися.
- Питання сформульовані українською, документація англійською, тому система має працювати між мовами. Це навмисне: так користуватиметься моя команда.

Фрагмент вважаємо **правильним**, якщо він належить правильній сторінці і містить доказ.

In [ ]:
BENCHMARK = [
    # (питання, правильні документи, regex-доказ у тексті фрагмента, еталонна відповідь)
    ("Якою командою запускають застосунок FastAPI у режимі розробки?", ["fastapi-cli"], r"you can use the `fastapi dev` command", "Командою `fastapi dev`."),
    ("Чим `fastapi run` відрізняється від `fastapi dev` за замовчуванням?", ["fastapi-cli"], r"By default, \*\*auto-reload\*\* is disabled", "`fastapi run` — режим production: автоперезавантаження вимкнене, слухає 0.0.0.0; `fastapi dev` — автоперезавантаження увімкнене, слухає 127.0.0.1."),
    ("Яке значення змінної середовища FASTAPI_ENV встановлює команда `fastapi dev`?", ["fastapi-cli"], r"sets the `FASTAPI_ENV` environment variable to `development`", "development."),
    ("Яким параметром Query обмежують максимальну довжину рядкового параметра?", ["tutorial/query-params-str-validations"], r"set the parameter `max_length` to `50`", "Параметром `max_length`."),
    ("Який код статусу повертає FastAPI, якщо query-параметр має невалідне значення?", ["tutorial/query-params", "tutorial/handling-errors"], r"`422`", "422 (помилка валідації)."),
    ("Який виняток треба викликати, щоб повернути клієнту HTTP-помилку, наприклад 404?", ["tutorial/handling-errors"], r"you use `HTTPException`", "`HTTPException`."),
    ("За якою адресою за замовчуванням доступна інтерактивна документація Swagger UI?", ["tutorial/first-steps"], r"Now go to http://127\.0\.0\.1:8000/docs", "За адресою /docs (http://127.0.0.1:8000/docs)."),
    ("Як вимкнути автоматичну документацію Swagger UI?", ["tutorial/metadata"], r"disable it by setting `docs_url=None`", "Передати `docs_url=None` у FastAPI()."),
    ("У якому порядку треба оголосити шляхи /users/me і /users/{user_id}?", ["tutorial/path-params"], r"declared before", "Спочатку /users/me, а потім /users/{user_id}."),
    ("Якою опцією запускають кілька процесів-воркерів сервера?", ["deployment/server-workers"], r"start multiple workers with the `--workers` command line option", "Опцією `--workers`, наприклад `--workers 4`."),
    ("Який клас використовують, щоб виконати код у фоні після відправлення відповіді?", ["tutorial/background-tasks"], r"First, import `BackgroundTasks`", "`BackgroundTasks`."),
    ("Який пакет треба встановити, щоб отримувати дані HTML-форм?", ["tutorial/request-forms", "tutorial/request-files", "tutorial/request-form-models", "tutorial/request-forms-and-files"], r"first install `python-multipart`", "`python-multipart`."),
    ("Чому для завантаження файлів зручніше UploadFile, ніж bytes?", ["tutorial/request-files"], r"spooled", "UploadFile використовує «spooled» файл: зберігає в пам'яті до певного ліміту, далі на диску, тому підходить для великих файлів і не вичерпує пам'ять."),
    ("Який HTTP-метод використовують для часткового оновлення даних?", ["tutorial/body-updates"], r"HTTP `PATCH` operation to \*partially\* update", "`PATCH`."),
    ("Який пакет потрібен для використання TestClient?", ["tutorial/testing"], r"first install `httpx`", "`httpx`."),
    ("Яку бібліотеку рекомендує документація для хешування паролів у прикладі з JWT?", ["tutorial/security/oauth2-jwt"], r"pwdlib is a great Python package", "`pwdlib` (з алгоритмом Argon2)."),
    ("Яким алгоритмом підписують JWT-токен у прикладі з документації?", ["tutorial/security/oauth2-jwt"], r"ALGORITHM. with the algorithm used to sign the JWT token and set it to .\"HS256\"", "HS256."),
    ("Яке значення за замовчуванням має параметр allow_methods у CORSMiddleware?", ["tutorial/cors"], r"allow_methods. - A list of HTTP methods.*Defaults to", "`['GET']`."),
    ("Що рекомендують робити, якщо не знаєш, чи використовувати async def, чи звичайний def?", ["async"], r"If you just don't know, use normal", "Використовувати звичайний `def`."),
    ("Яку команду вказують у CMD у Dockerfile для запуску застосунку FastAPI?", ["deployment/docker"], r'CMD \["fastapi", "run"', 'CMD ["fastapi", "run", "app/main.py", "--port", "80"].'),
    ("Скільки процесів зазвичай запускають в одному контейнері?", ["deployment/docker"], r"normally has a \*\*single process\*\*", "Один процес."),
    ("Який декоратор застосовують, щоб створити middleware?", ["tutorial/middleware"], r'@app\.middleware\("http"\)', '`@app.middleware("http")`.'),
    ("Який параметр OAuth2PasswordBearer задає URL, за яким клієнт отримує токен?", ["tutorial/security/first-steps"], r"we pass in the `tokenUrl` parameter", "`tokenUrl`."),
    ("Який пакет потрібен для читання налаштувань застосунку зі змінних середовища?", ["advanced/settings"], r"Install `pydantic-settings`", "`pydantic-settings`."),
    # питання без відповіді в документах
    ("Як налаштувати обмеження частоти запитів (rate limiting) у FastAPI?", [], None, "немає відповіді"),
    ("Як експортувати метрики застосунку FastAPI в Prometheus?", [], None, "немає відповіді"),
    ("Як підключити MongoDB через Beanie ODM у FastAPI?", [], None, "немає відповіді"),
    ("Скільки коштує розгортання застосунку у FastAPI Cloud?", [], None, "немає відповіді"),
]


NO_ANSWER = [b for b in BENCHMARK if not b[1]]
answerable = [(q, gold, ev) for q, gold, ev, ref in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(NO_ANSWER)}")

# Перевірка бенчмарку: кожен доказ справді є в тексті правильної сторінки
by_id = {d["doc_id"]: d for d in docs}
for q, gold, ev in answerable:
    assert any(re.search(ev, re.sub(r"\s+", " ", by_id[g]["text"])) for g in gold), q
print("Усі докази знайдено в правильних сторінках корпусу.")

Питань: 28, з них без відповіді в документах: 4
Усі докази знайдено в правильних сторінках корпусу.


## 1.3. Індексація

Обрані рішення та їх обґрунтування:

- **Модель ембедингів — Qwen/Qwen3-Embedding-0.6B** (як у демо). Вона багатомовна (враховує питання українською й текст англійською), приймає довгі фрагменти й має окремий префікс для питань; вектори нормалізовані, тож косинус дорівнює скалярному добутку.
- **Розбиття — RecursiveCharacterTextSplitter** за ієрархією роздільників: порожній рядок (абзац) → рядок → речення → слово. Так абзаци та блоки коду розрізаються в останню чергу. До кожного фрагмента при індексації додається назва сторінки (вона містить тему, якої може не бути в самому фрагменті).
- **Розмір фрагмента й перекриття.** Порівнюємо дві конфігурації: **500/100** і **1000/200** символів. Менші фрагменти точніші, більші містять більше контексту. Перекриття більше за довжину регулярного виразу-доказу, тому фраза-доказ не втрачається на межі фрагментів.
- **Векторне сховище — FAISS** (IndexFlatIP, пошук за скалярним добутком). Для кількох тисяч векторів точний повний перебір миттєвий, тож наближені індекси не потрібні. Окремо будуємо BM25 для гібридного пошуку.

In [ ]:
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=DEVICE)
embedder.max_seq_length = 512
if DEVICE == "cuda":
    embedder.half()


def split_documents(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                              separators=["\n\n", "\n", ". ", " ", ""])
    chunks = []
    for d in docs:
        for piece in splitter.split_text(d["text"]):
            chunks.append({"id": len(chunks), "doc_id": d["doc_id"], "title": d["title"], "url": d["url"], "text": piece})
    return chunks


def tokenize(text):
    return re.findall(r"\w+", text.lower())


def build_index(chunk_size, chunk_overlap):
    chunks = split_documents(docs, chunk_size, chunk_overlap)
    texts = [f'{c["title"]}\n\n{c["text"]}' for c in chunks]
    t0 = time.time()
    vectors = embedder.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).astype("float32")
    faiss_index = faiss.IndexFlatIP(vectors.shape[1])
    faiss_index.add(vectors)
    bm25 = BM25Okapi([tokenize(t) for t in texts])
    lengths = [len(c["text"]) for c in chunks]
    print(f"Розбиття {chunk_size}/{chunk_overlap}: {len(chunks)} фрагментів, середня довжина {np.mean(lengths):.0f} символів, "
          f"вектори {vectors.shape}, індексація {time.time() - t0:.0f} с")
    return {"name": f"{chunk_size}/{chunk_overlap}", "chunks": chunks, "faiss": faiss_index, "bm25": bm25}


idx500 = build_index(500, 100)
idx1000 = build_index(1000, 200)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Розбиття 500/100: 2605 фрагментів, середня довжина 404 символів, вектори (2605, 1024), індексація 33 с
Розбиття 1000/200: 1289 фрагментів, середня довжина 862 символів, вектори (1289, 1024), індексація 33 с


## 1.4. Ретрив

Пошук реалізовано в чотирьох варіантах, які ми зіставляємо:
1. **Векторний** (FAISS, косинусна подібність).
2. **BM25** за лексичним збігом. Питання українською, а текст англійською, тому BM25 працює лише на збігах назв (fastapi dev, HTTPException, FASTAPI_ENV) і від початку очікувано слабкий.
3. **Гібридний:** векторний + BM25, об'єднані Reciprocal Rank Fusion (RRF, c = 60).
4. **Гібридний + cross-encoder reranker** BAAI/bge-reranker-v2-m3 для 20 кандидатів.

Метрики: **hit@k** (частка питань, для яких правильний фрагмент є серед перших k) і **MRR** (середнє значення 1/ранг). Оцінюємо 24 питання з відповіддю.

In [ ]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=DEVICE)


def vector_search(idx, query, k):
    q = embedder.encode([query], prompt_name="query", normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, ids = idx["faiss"].search(q, k)
    return [idx["chunks"][i] for i in ids[0]]


def bm25_search(idx, query, k):
    scores = idx["bm25"].get_scores(tokenize(query))
    return [idx["chunks"][i] for i in np.argsort(-scores)[:k] if scores[i] > 0]


def rrf(rankings, c=60):
    scores = {}
    for ranking in rankings:
        for rank, ch in enumerate(ranking, start=1):
            scores[ch["id"]] = scores.get(ch["id"], 0) + 1 / (c + rank)
    return sorted(scores, key=scores.get, reverse=True)


def hybrid_search(idx, query, k, pool=20):
    ids = rrf([vector_search(idx, query, pool), bm25_search(idx, query, pool)])
    return [idx["chunks"][i] for i in ids[:k]]


def rerank_search(idx, query, k, pool=20):
    cands = hybrid_search(idx, query, pool, pool)
    scores = reranker.predict([(query, f'{c["title"]}\n\n{c["text"]}') for c in cands], batch_size=16, show_progress_bar=False)
    return [cands[i] for i in np.argsort(-scores)[:k]]


def norm(text):
    return re.sub(r"\s+", " ", text)


def relevant(chunk, gold, evidence):
    return chunk["doc_id"] in gold and re.search(evidence, norm(chunk["text"])) is not None


def first_hit_rank(results, gold, evidence):
    for rank, ch in enumerate(results, start=1):
        if relevant(ch, gold, evidence):
            return rank
    return None


def hit_rate(ranks, k):
    return float(np.mean([r is not None and r <= k for r in ranks]))


def mrr(ranks):
    return float(np.mean([1 / r if r else 0.0 for r in ranks]))


print("Демо: питання «Яке значення FASTAPI_ENV встановлює fastapi dev?»")
for ch in rerank_search(idx1000, BENCHMARK[2][0], 3):
    print("  ", ch["doc_id"], "|", repr(ch["text"][:90]))

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Демо: питання «Яке значення FASTAPI_ENV встановлює fastapi dev?»
   fastapi-cli | '## `fastapi dev`\n\nRunning `fastapi dev` initiates development mode.\n\nBy default, **auto-re'
   tutorial/frontend | 'So, in most cases you can use `app.frontend("/", directory="dist")` without specifying the'
   environment-variables | '# Environment Variables\n\nAn **environment variable** (also known as an **env var**) is a v'


In [ ]:
CONFIGS = {
    "вектор, 500/100": (idx500, vector_search),
    "вектор, 1000/200": (idx1000, vector_search),
    "BM25, 1000/200": (idx1000, bm25_search),
    "гібрид (RRF), 1000/200": (idx1000, hybrid_search),
    "гібрид + reranker, 500/100": (idx500, rerank_search),
    "гібрид + reranker, 1000/200": (idx1000, rerank_search),
}
rows, all_ranks = [], {}
for name, (idx, fn) in CONFIGS.items():
    t0 = time.time()
    ranks = [first_hit_rank(fn(idx, q, 10), gold, ev) for q, gold, ev in answerable]
    all_ranks[name] = ranks
    rows.append({"конфігурація": name, "hit@1": hit_rate(ranks, 1), "hit@3": hit_rate(ranks, 3), "hit@5": hit_rate(ranks, 5),
                 "hit@10": hit_rate(ranks, 10), "MRR": mrr(ranks), "с на питання": (time.time() - t0) / len(answerable)})
retrieval_table = pd.DataFrame(rows).set_index("конфігурація").round(2)
print(retrieval_table.to_string())

                             hit@1  hit@3  hit@5  hit@10   MRR  с на питання
конфігурація                                                                
вектор, 500/100               0.33   0.54   0.62    0.75  0.45          0.04
вектор, 1000/200              0.38   0.67   0.79    0.83  0.53          0.04
BM25, 1000/200                0.17   0.29   0.38    0.46  0.26          0.00
гібрид (RRF), 1000/200        0.29   0.50   0.62    0.71  0.42          0.05
гібрид + reranker, 500/100    0.58   0.75   0.75    0.79  0.67          0.76
гібрид + reranker, 1000/200   0.54   0.75   0.79    0.92  0.64          1.37


### Результати ретриву

Метрики: hit@k (частка питань, для яких правильна сторінка є серед k перших фрагментів) і MRR, 24 питання з відповіддю.

Що видно в таблиці:
- **Reranker дає найбільший приріст.** Для 500/100 hit@1 зростає з 0.33 (вектор) до 0.58 (гібрид + reranker), MRR з 0.45 до 0.67. Cross-encoder читає питання й фрагмент разом, тому краще ранжує кандидатів, ніж косинусна близькість.
- **Гібрид без reranker не кращий за чистий вектор** (MRR 0.42 проти 0.45). Причина в мовах: питання українською, документація англійською, тому BM25 збігається майже лише на латинських ідентифікаторах (`Query`, `max_length`). Окремо BM25 слабкий (MRR 0.26), і RRF підмішує його шум.
- **Більші фрагменти допомагають вектору.** З 1000/200 вектор має hit@5 0.79 проти 0.62 для 500/100, бо в довшому фрагменті більше контексту. З reranker різниця менша: 500/100 краще за hit@1 і MRR, 1000/200 краще за hit@10 (0.92).
- Для генерації взято **гібрид + reranker, 500/100, k=5**: дрібні фрагменти дають точніші цитати й коротший контекст для моделі.

## 1.5. Генерація з цитуванням

**Генеративна модель** — локальна Qwen/Qwen3-4B-Instruct-2507 (fp16 на GPU T4), та сама, що в демо 2 і 3; запускається прямо в ноутбуці через transformers. Детермінована генерація (без семплінгу).

Системний промпт вимагає: (1) відповідати українською **лише на основі наданих фрагментів**, (2) після кожного твердження вказувати номер фрагмента у квадратних дужках, (3) якщо відповіді у фрагментах немає, відповісти фіксованою фразою відмови.

Оцінюємо дві системи на всіх 28 питаннях:
- **базова:** векторний пошук, фрагменти 500/100, в контекст потрапляють 3 фрагменти;
- **фінальна:** найкраща за MRR конфігурація з таблиці 1.4, в контекст потрапляють 5 фрагментів.

Метрики: частка правильних відповідей на питання з відповіддю (судить LLM-суддя за еталоном; та сама модель, як і в демо 3, зі зрозумілими обмеженнями), **коректність цитування** (серед відповідей без відмови: чи правильний фрагмент входить до процитованих) і частка правильних відмов на питання без відповіді (визначається за фразою відмови).

**Чому не OpenAI-сумісний ендпойнт.** У методичці дозволено локальну модель або API. Ендпойнт із лабораторної 4 у Colab не піднято, тому модель запускається локально в самому ноутбуці через transformers. Щоб перейти на ендпойнт, достатньо замінити виклик у функції генерації на запит до OpenAI-сумісного API; решта конвеєра (пошук, промпт, оцінка) не змінюється.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(GEN_NAME)
try:
    llm = AutoModelForCausalLM.from_pretrained(GEN_NAME, dtype=torch.float16, device_map="cuda")
except TypeError:
    llm = AutoModelForCausalLM.from_pretrained(GEN_NAME, torch_dtype=torch.float16, device_map="cuda")
llm.eval()

REFUSAL = "У наданих документах немає відповіді."
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання про веб-фреймворк FastAPI. "
    "Відповідай українською лише на основі наданих фрагментів документації (вони англійською). "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Назви функцій, класів, параметрів і команди залишай у вихідному вигляді. "
    f"Якщо у фрагментах немає відповіді на питання, відповідай лише: «{REFUSAL}» "
    "Не використовуй знання, яких немає у фрагментах."
)
JUDGE_PROMPT = (
    "Ти перевіряєш відповіді системи на питання. Порівняй відповідь системи з еталоном. "
    "Відповідь правильна, якщо за змістом збігається з еталоном (додаткові правильні деталі припустимі). "
    "Відповідай одним словом: ПРАВИЛЬНО або НЕПРАВИЛЬНО."
)


def chat(system, user, max_new_tokens=300):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def answer_question(question, context_chunks):
    context = "\n\n".join(f'[{i}] {c["title"]} ({c["doc_id"]})\n{c["text"]}' for i, c in enumerate(context_chunks, start=1))
    return chat(SYSTEM_PROMPT, f"Фрагменти документації:\n\n{context}\n\nПитання: {question}")


def judge(question, reference, answer):
    verdict = chat(JUDGE_PROMPT, f"Питання: {question}\nЕталон: {reference}\nВідповідь системи: {answer}\nВердикт:", max_new_tokens=8).upper()
    return verdict.startswith("ПРАВИЛЬНО")


print(answer_question(BENCHMARK[2][0], rerank_search(idx1000, BENCHMARK[2][0], 5)))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Команда `fastapi dev` встановлює значення змінної середовища `FASTAPI_ENV` у `development`. [1]


In [ ]:
def cited_numbers(answer):
    return sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})


def run_system(name, idx, search_fn, k):
    t0 = time.time()
    rows = []
    for q, gold, ev, ref in BENCHMARK:
        ctx = search_fn(idx, q, k)
        ans = answer_question(q, ctx)
        refused = "немає відповіді" in ans.lower()
        cited = [ctx[n - 1] for n in cited_numbers(ans) if 1 <= n <= len(ctx)]
        if gold:
            ok = (not refused) and judge(q, ref, ans)
            cite_ok = None if refused else any(relevant(c, gold, ev) for c in cited)
            in_ctx = any(relevant(c, gold, ev) for c in ctx)
        else:
            ok, cite_ok, in_ctx = refused, None, None
        rows.append({"система": name, "питання": q, "є відповідь": bool(gold), "еталон": ref, "відповідь": ans, "правильно": ok,
                     "цитування": cite_ok, "доказ у контексті": in_ctx, "відмова": refused, "знайдено": [c["doc_id"] for c in ctx]})
    df = pd.DataFrame(rows)
    has = df[df["є відповідь"]]
    cites = has["цитування"].dropna().astype(bool)
    print(f"[{name}] {time.time() - t0:.0f} с")
    print(f"  Правильних відповідей на питання з відповіддю: {has['правильно'].mean():.0%} ({int(has['правильно'].sum())} з {len(has)})")
    print(f"  Коректних цитувань (серед відповідей без відмови): {cites.mean():.0%} ({int(cites.sum())} з {len(cites)})")
    print(f"  Правильних відмов на питання без відповіді: {df[~df['є відповідь']]['правильно'].mean():.0%}")
    return df


best_name = retrieval_table.sort_values(["MRR", "hit@5"], ascending=False).index[0]
print("Найкраща конфігурація ретриву за MRR:", best_name)
base_idx, base_fn = CONFIGS["вектор, 500/100"]
best_idx, best_fn = CONFIGS[best_name]
eval_base = run_system("базова: вектор 500/100, k=3", base_idx, base_fn, 3)
eval_best = run_system(f"фінальна: {best_name}, k=5", best_idx, best_fn, 5)

Найкраща конфігурація ретриву за MRR: гібрид + reranker, 500/100
[базова: вектор 500/100, k=3] 112 с
  Правильних відповідей на питання з відповіддю: 58% (14 з 24)
  Коректних цитувань (серед відповідей без відмови): 81% (13 з 16)
  Правильних відмов на питання без відповіді: 100%
[фінальна: гібрид + reranker, 500/100, k=5] 183 с
  Правильних відповідей на питання з відповіддю: 83% (20 з 24)
  Коректних цитувань (серед відповідей без відмови): 65% (15 з 23)
  Правильних відмов на питання без відповіді: 75%


### Результати генерації

Порівняно два повні RAG-системи на 28 питаннях (24 з відповіддю, 4 без відповіді в корпусі). Генератор — Qwen3-4B-Instruct-2507, оцінювач — та сама модель (LLM-judge) разом із перевіркою фрази відмови.

| Система | Правильні відповіді | Коректні цитування | Відмови на питання без відповіді | Час |
|---|---|---|---|---|
| Базова: вектор 500/100, k=3 | 58% (14/24) | 81% (13/16) | 100% | 112 с |
| Фінальна: гібрид + reranker 500/100, k=5 | 83% (20/24) | 65% (15/23) | 75% | 183 с |

Висновки:
- Фінальна система відповідає правильно значно частіше (+25 п.п.), бо reranker частіше кладе потрібний фрагмент у контекст. Базова система часто хибно відмовляється («У наданих документах немає відповіді»), коли доказу немає в трьох фрагментах.
- Цитування в фінальній системі гірші. Метрика вважає цитату правильною, якщо цитований фрагмент містить доказ за регулярним виразом. Багато правильних відповідей отримують «хибне цитування» через вузький регулярний вираз або посилання на сусідній фрагмент тієї ж сторінки, тож цифра занижена.
- Базова система відмовилась на всіх 4 питаннях без відповіді, але частково тому, що відмовлялась і на питаннях з відповіддю. Фінальна на одне питання (про Prometheus) відповіла вигадкою замість відмови.
- Фінальна система повільніша на 70 с (reranker на 20 кандидатах для кожного питання).

## 1.6. Аналіз відмов

Нижче виведено всі випадки, коли система відповіла неправильно, не мала правильного фрагмента в контексті, процитувала не те джерело або відмовилася без підстав, для обох систем. Для кожного автоматично підказано ймовірний тип відмови, а нижче в тексті подано мій розбір із тим, що можна змінити.

In [ ]:
def hint(r):
    if not r["є відповідь"]:
        return "галюцинація: відповідь на питання без відповіді в документах"
    if r["відмова"]:
        return "хибна відмова" + (" попри правильний контекст (надмірна обережність моделі)" if r["доказ у контексті"] else ": доказу в контексті немає, промах ретриву")
    if not r["доказ у контексті"]:
        return "промах ретриву: правильного фрагмента немає в контексті"
    return "помилка генерації/цитування або помилка судді: правильний фрагмент був у контексті"


failures = pd.concat([eval_base, eval_best])
failures = failures[(~failures["правильно"]) | (failures["цитування"] == False) | (failures["є відповідь"] & ~failures["доказ у контексті"].fillna(True).astype(bool))]
print(f"Проблемних випадків: {len(failures)} (базова: {(failures['система'] == eval_base['система'][0]).sum()}, фінальна: {(failures['система'] == eval_best['система'][0]).sum()})\n")
for _, r in failures.iterrows():
    print(f"[{r['система']}]")
    print("  питання:", r["питання"])
    print("  еталон:", r["еталон"])
    print("  відповідь:", r["відповідь"][:400].replace("\n", " "))
    print("  знайдено:", r["знайдено"], "| правильно:", r["правильно"], "| цитування:", r["цитування"], "| доказ у контексті:", r["доказ у контексті"])
    print("  ймовірний тип:", hint(r), "\n")

Проблемних випадків: 23 (базова: 11, фінальна: 12)

[базова: вектор 500/100, k=3]
  питання: Яким параметром Query обмежують максимальну довжину рядкового параметра?
  еталон: Параметром `max_length`.
  відповідь: Параметром Query, яким обмежують максимальну довжину рядкового параметра, є `max_length`. [1]
  знайдено: ['tutorial/query-params-str-validations', 'tutorial/query-params-str-validations', 'tutorial/query-params-str-validations'] | правильно: True | цитування: False | доказ у контексті: False
  ймовірний тип: промах ретриву: правильного фрагмента немає в контексті 

[базова: вектор 500/100, k=3]
  питання: Який код статусу повертає FastAPI, якщо query-параметр має невалідне значення?
  еталон: 422 (помилка валідації).
  відповідь: У наданих документах немає відповіді.
  знайдено: ['tutorial/query-params-str-validations', 'tutorial/query-params-str-validations', 'tutorial/query-params-str-validations'] | правильно: False | цитування: None | доказ у контексті: False
  ймовірний

/tmp/ipykernel_2215/3707989233.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  failures = failures[(~failures["правильно"]) | (failures["цитування"] == False) | (failures["є відповідь"] & ~failures["доказ у контексті"].fillna(True).astype(bool))]


### Аналіз відмов

Скрипт вище знайшов 23 проблемні випадки (базова система — 11, фінальна — 12). Нижче шість типових, кожен із типом і виправленням.

| № | Питання (скорочено) | Що сталося | Тип відмови | Як виправити |
|---|---|---|---|---|
| 1 | Який код статусу при невалідному query-параметрі? (базова) | Відповідь «немає відповіді», у трьох фрагментах лише сторінка про валідацію рядків, сторінки про помилки немає | Промах ретриву з хибною відмовою | Взяти більше фрагментів (k=5–10) і reranker; розширювати питання переформулюванням (query rewriting) |
| 2 | Яку бібліотеку для хешування паролів рекомендує документація в прикладі з JWT? (базова) | Модель відповіла «`pyjwt`»; правильно `pwdlib` (Argon2). Сторінка про OAuth2 і JWT знайдена, але модель сплутала бібліотеку для токенів із бібліотекою для хешування паролів | Галюцинація / плутанина в контексті | Вимагати від моделі цитувати дослівний фрагмент; перевіряти, що названий пакет є в контексті біля слова «hash»; більша модель |
| 3 | Чому UploadFile зручніший за bytes? (обидві системи) | Відповідь загальна («гнучкіший»), без суті про spooled-файл (пам'ять до ліміту, далі диск) | Неповна відповідь: потрібне речення не потрапило в контекст | Більші фрагменти (1000/200) або розширення контексту сусідніми фрагментами |
| 4 | Що робити, якщо не знаєш, async def чи def? (фінальна) | Модель порадила `async def`, у документації радять звичайний `def`; цитата [1] при цьому є | Помилка генерації попри контекст | Нижча температура вже стоїть (greedy), тому потрібен чіткіший промпт («відповідай лише з тексту») і перевірка відповіді другим проходом |
| 5 | Яку команду вказують у CMD у Dockerfile? (фінальна) | Додано зайвий `--workers 4`, ймовірно з іншого прикладу в контексті | Змішування джерел (контекст містить два різні приклади) | Давати кожному фрагменту окремий номер і просити відповідати з одного джерела; зменшити k |
| 6 | Як експортувати метрики у Prometheus? (фінальна, питання без відповіді) | Замість відмови модель вигадала налаштування OpenTelemetry | Галюцинація на питанні без відповіді | Поріг релевантності reranker: якщо найкращий score нижче порога, відмовляти без звернення до моделі |

Окремо: частина «відмов» у таблиці скрипта є **артефактом метрики** (правильна відповідь із `cit: False`, бо регулярний вираз доказу вузький). Це видно в питаннях про max_length, адресу Swagger UI та воркери: відповіді правильні, а метрика цитування їх не зараховує. Виправлення: широчі регулярні вирази або перевірка цитування LLM-суддею.

Тип «переповнення контексту» тут не зустрівся: при k=5 і фрагментах по 500 символів контекст становить близько 2500 символів, що значно менше за вікно моделі. Він став би проблемою при великих k або фрагментах 1000/200 разом із довгими прикладами коду.

## 1.7. Демонстрація

Простий інтерфейс на **Gradio**: поле для питання, відповідь і список джерел (назва сторінки та посилання на неї в документації). Використовується фінальна система. Під полем є **готові приклади питань**: п’ять із відповіддю в документації й три, на які система має відмовитися (відповіді в корпусі немає або питання не про FastAPI). Ті самі питання нижче прогнано в коді, щоб результат лишався в ноутбуці й після закриття публічного посилання Gradio.

In [ ]:
import gradio as gr

EXAMPLES = [
    "Як запустити застосунок FastAPI для розробки?",
    "Як задати мінімальну довжину рядкового query-параметра?",
    "Як запустити кілька воркерів на production?",
    "Для чого в FastAPI потрібен Depends?",
    "Як підключити GraphQL-схему на Strawberry до застосунку?",
    "Як налаштувати кешування відповідей у Redis?",
    "Як підключити FastAPI до Kafka-брокера повідомлень?",
    "Яка столиця Франції?",
]


def ask_ui(question):
    ctx = best_fn(best_idx, question, 5)
    answer = answer_question(question, ctx)
    sources = "\n".join(f'[{i}] {c["title"]} — {c["url"]}' for i, c in enumerate(ctx, start=1))
    return answer, sources


demo = gr.Interface(fn=ask_ui, inputs=gr.Textbox(label="Питання про FastAPI", lines=2),
                    outputs=[gr.Textbox(label="Відповідь", lines=6), gr.Textbox(label="Джерела", lines=6)],
                    examples=[[q] for q in EXAMPLES], cache_examples=False,
                    title="RAG за документацією FastAPI")
try:
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio не запустився:", e)

for question in EXAMPLES:
    answer, sources = ask_ui(question)
    print("Питання:", question, "\nВідповідь:", answer, "\nДжерела:\n" + sources, "\n")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d021988dbed0aad1f4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Питання: Як запустити застосунок FastAPI для розробки? 
Відповідь: Використовуйте команду `fastapi dev` для запуску застосунку FastAPI для розробки [2]. 
Джерела:
[1] First Steps — https://fastapi.tiangolo.com/tutorial/first-steps/
[2] FastAPI CLI — https://fastapi.tiangolo.com/fastapi-cli/
[3] First Steps — https://fastapi.tiangolo.com/tutorial/first-steps/
[4] First Steps — https://fastapi.tiangolo.com/tutorial/first-steps/
[5] FastAPI Cloud — https://fastapi.tiangolo.com/deployment/fastapicloud/ 

Питання: Як задати мінімальну довжину рядкового query-параметра? 
Відповідь: Мінімальну довжину рядкового query-параметра можна задати за допомогою параметра `min_length` у визначенні query-параметра через `Query`. Наприклад, `min_length=3` вказує, що довжина рядка має бути не меншою за 3 символи [2]. 
Джерела:
[1] Query Parameters and String Validations — https://fastapi.tiangolo.com/tutorial/query-params-str-validations/
[2] Query Parameters and String Validations — https://fastapi.tiang

## 1.8. Мій звіт

### 1. Корпус і чому потрібен RAG
Корпус — офіційна документація Python-фреймворку **FastAPI** (англійською), 117 сторінок (≈976 тис. символів, ≈244 тис. токенів), завантажена з зафіксованого коміту репозиторію (ліцензія MIT). Це предметна область мого дипломного проєкту. Питання ставляться українською, тож перевіряється й міжмовний пошук. RAG потрібен, бо документація велика й постійно змінюється (наприклад, нові версії рекомендують інші бібліотеки), модель без пошуку відповідає з застарілих або вигаданих даних, а з RAG відповідь спирається на конкретну сторінку з посиланням.

### 2. Бенчмарк
28 питань: 24 мають відповідь у корпусі (для кожного задано правильні сторінки, регулярний вираз доказу й еталонну відповідь), 4 навмисно без відповіді (перевірка відмови). Є питання на факт, на процедуру («як зробити…»), на порівняння та «пастки», де в моделі може бути застаріла відповідь (бібліотека для хешування паролів).

### 3. Індексація
- Модель ембедингів: Qwen3-Embedding-0.6B (fp16, довжина до 512 токенів), запити з префіксом-інструкцією.
- Розбиття: RecursiveCharacterTextSplitter у двох конфігураціях: 500/100 (2605 фрагментів, у середньому 404 символи) і 1000/200 (1289 фрагментів, 862 символи). Заголовок сторінки додається перед ембедингом.
- Сховище: FAISS IndexFlatIP (косинусна близькість), плюс BM25 для лексичного пошуку. Індекс будується в ноутбуці й у репозиторій не додається.

### 4. Результати ретриву для різних конфігурацій
| Конфігурація | hit@1 | hit@3 | hit@5 | hit@10 | MRR |
|---|---|---|---|---|---|
| вектор 500/100 | 0.33 | 0.54 | 0.62 | 0.75 | 0.45 |
| вектор 1000/200 | 0.38 | 0.67 | 0.79 | 0.83 | 0.53 |
| BM25 | 0.17 | 0.29 | 0.38 | 0.46 | 0.26 |
| гібрид (RRF) | 0.29 | 0.50 | 0.62 | 0.71 | 0.42 |
| гібрид + reranker 500/100 | 0.58 | 0.75 | 0.75 | 0.79 | 0.67 |
| гібрид + reranker 1000/200 | 0.54 | 0.75 | 0.79 | 0.92 | 0.64 |

Reranker (BAAI/bge-reranker-v2-m3) дає найбільший приріст. BM25 слабкий через різні мови питання й документів, тому гібрид без reranker гірший за чистий вектор.

### 5. Генеративна модель і результати відповідей
Генератор — Qwen3-4B-Instruct-2507 (fp16, greedy). Промпт вимагає відповідати лише з наданих фрагментів, цитувати їх як [n] і відмовлятися фразою «У наданих документах немає відповіді.». Базова система (вектор 500/100, k=3) дала 58% правильних відповідей, фінальна (гібрид + reranker 500/100, k=5) — 83%. Цитування: 81% проти 65% (метрика занижена через вузькі регулярні вирази). Відмови на питаннях без відповіді: 100% проти 75%.

### 6. Аналіз відмов
Знайдено 23 проблемні випадки. Основні типи: промах ретриву (потрібного фрагмента немає в контексті), хибна відмова базової системи, змішування джерел і помилки генерації попри правильний контекст, одна галюцинація на питанні без відповіді (Prometheus) і артефакти метрики. Запропоновані виправлення: query rewriting, більші фрагменти або сусідні фрагменти, поріг релевантності reranker для відмови, суворіший промпт, оцінка цитування LLM-суддею.

### 7. Висновок
Для документації FastAPI найкраще працює схема «векторний пошук + BM25 + cross-encoder reranker»: правильних відповідей 83% проти 58% у простого векторного RAG. Головні обмеження: міжмовний пошук (BM25 майже не допомагає), мала модель-генератор (4B) і галюцинації на питаннях без відповіді. Далі варто додати поріг відмови за score reranker, переформулювання запитів англійською перед пошуком і ширшу оцінку цитувань. Демонстрація (розділ 1.7) показує відповідь і джерела для кількох прикладів.